# Transações distribuídas na prática: a compra do tênis

Este notebook acompanha o tópico **Transações Distribuídas (2PC e Saga)** do Guia Aberto de Arquitetura de Software e usa o mesmo exemplo do texto: uma compra de tênis que passa por três serviços, **Estoque**, **Pagamento** e **Frete**.

Vamos fazer três coisas:

1. **Implementação correta:** a Saga com compensação, que desfaz os passos já feitos quando um deles falha.
2. **Violação:** a mesma Saga, mas sem as compensações.
3. **Consequência mensurável:** o que sobra de errado no sistema, e quanto o 2PC custa quando muita gente disputa o mesmo tênis.

Só usamos a biblioteca padrão do Python. **Atenção:** as latências abaixo são parâmetros de uma simulação (`time.sleep`), não medidas de um sistema real. O que vale é o comportamento, não os números exatos.

Para rodar tudo: *Ambiente de execução → Executar tudo* (leva menos de um minuto).

In [ ]:
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor

# Tempo de cada serviço para fazer o seu trabalho (segundos).
# O Frete (Correios) é de propósito o mais lento.
LAT = {"estoque": 0.005, "pagamento": 0.020, "frete": 0.080}
PRECO = 100  # preço do tênis, em reais

## Parte A: o custo do 2PC quando todo mundo quer o mesmo tênis

Imagine o tênis em promoção: todos os pedidos disputam o **mesmo item do estoque**. Modelamos isso com um único `Lock`.

- No **2PC**, o item fica travado durante as duas fases. Cada fase espera o participante mais lento (o Frete), então o lock é segurado por cerca de `2 × 80 ms`.
- Na **Saga**, o lock existe só dentro da transação local do estoque (`5 ms`). Pagamento e Frete rodam depois, sem segurar o item.

In [ ]:
def pedido_2pc(lock_item):
    with lock_item:                        # o item fica travado até a decisão final
        time.sleep(max(LAT.values()))      # fase 1 (prepare): espera o mais lento
        time.sleep(max(LAT.values()))      # fase 2 (commit): espera o mais lento


def pedido_saga(lock_item):
    with lock_item:                        # trava só durante a transação local
        time.sleep(LAT["estoque"])
    time.sleep(LAT["pagamento"])           # sem segurar o item do estoque
    time.sleep(LAT["frete"])


def rodar(modelo, n_pedidos, concorrencia):
    lock_item = threading.Lock()
    duracoes = []

    def um_pedido(_):
        t0 = time.perf_counter()
        modelo(lock_item)
        duracoes.append(time.perf_counter() - t0)

    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concorrencia) as pool:
        list(pool.map(um_pedido, range(n_pedidos)))
    total = time.perf_counter() - inicio
    return n_pedidos / total, 1000 * sum(duracoes) / len(duracoes)


N = 30
print(f"{'concorrência':>12}  {'modelo':<6} {'pedidos/s':>10} {'latência média (ms)':>20}")
for conc in (1, 5, 20):
    for nome, modelo in (("2PC", pedido_2pc), ("Saga", pedido_saga)):
        vazao, latencia = rodar(modelo, N, conc)
        print(f"{conc:>12}  {nome:<6} {vazao:>10.1f} {latencia:>20.0f}")

## Parte B: a Saga com e sem compensação

Agora o Frete falha em parte dos pedidos (`taxa_falha=0.25`; com a semente fixa, 60 dos 200 pedidos falham, e todo mundo vê os mesmos pedidos falhando). Cada serviço mexe no seu pedaço do estado: o Estoque reserva, o Pagamento cobra, o Frete gera a etiqueta.

Três versões:

- **`saga` com `compensar=True`:** se um passo falha, desfaz os anteriores em ordem inversa.
- **`saga` com `compensar=False`:** a violação. Dá o pedido por cancelado, mas não desfaz nada.
- **`dois_pc`:** para comparar. Todos votam antes e nada é alterado se alguém votar não.

In [ ]:
class Falha(Exception):
    pass


class Loja:
    def __init__(self, estoque):
        self.estoque_inicial = estoque
        self.estoque = estoque
        self.cobrado = 0
        self.etiquetas = 0


# Cada serviço faz o seu trabalho (e sabe desfazê-lo).
def reservar(loja, pedido): loja.estoque -= 1
def devolver(loja, pedido): loja.estoque += 1
def cobrar(loja, pedido): loja.cobrado += PRECO
def estornar(loja, pedido): loja.cobrado -= PRECO


def etiquetar(loja, pedido):
    if pedido["falha_frete"]:
        raise Falha("Correios fora do ar")
    loja.etiquetas += 1


PASSOS = [(reservar, devolver), (cobrar, estornar), (etiquetar, None)]


def saga(loja, pedido, compensar=True):
    feitos = []                              # compensações dos passos já concluídos
    try:
        for passo, desfazer in PASSOS:
            passo(loja, pedido)
            feitos.append(desfazer)
    except Falha:
        if compensar:
            for desfazer in reversed(feitos):    # desfaz em ordem inversa
                desfazer(loja, pedido)
        return "cancelado"
    return "confirmado"


def dois_pc(loja, pedido):
    votos = [True, True, not pedido["falha_frete"]]   # fase 1: todos votam
    if not all(votos):
        return "cancelado"                            # abort: nada foi efetivado
    reservar(loja, pedido)                            # fase 2: todos efetivam
    cobrar(loja, pedido)
    etiquetar(loja, pedido)
    return "confirmado"

In [ ]:
def simular(modelo, n=200, taxa_falha=0.25, semente=7):
    rnd = random.Random(semente)             # mesma sequência de falhas em todas as versões
    loja = Loja(estoque=1000)
    resultados = [modelo(loja, {"falha_frete": rnd.random() < taxa_falha}) for _ in range(n)]
    confirmados = resultados.count("confirmado")
    estoque_preso = loja.estoque_inicial - loja.estoque - confirmados
    cobrado_indevido = loja.cobrado - confirmados * PRECO
    return confirmados, estoque_preso, cobrado_indevido


versoes = [
    ("Saga com compensação", lambda l, p: saga(l, p, compensar=True)),
    ("Saga sem compensação", lambda l, p: saga(l, p, compensar=False)),
    ("2PC", dois_pc),
]

print(f"{'versão':<22} {'confirmados':>12} {'estoque preso':>14} {'cobrado a mais (R$)':>20}")
for nome, modelo in versoes:
    confirmados, preso, indevido = simular(modelo)
    print(f"{nome:<22} {confirmados:>12} {preso:>14} {indevido:>20}")
    if nome != "Saga sem compensação":
        assert preso == 0 and indevido == 0, "esta versão deveria terminar consistente"

## O que observar

- **Parte A:** com uma só thread (sem disputa), as duas abordagens andam parecidas. Quando a concorrência sobe, o 2PC **não melhora**: ele fica preso no tempo em que segura o item, e só um pedido avança por vez. A Saga passa a atender muito mais pedidos por segundo.
- **Parte B:** a Saga com compensação e o 2PC terminam **consistentes** (nenhuma unidade presa, nenhum real cobrado a mais). A Saga sem compensação diz "cancelado", mas deixa o estoque preso e o cliente cobrado por um tênis que nunca será enviado. E nenhum teste do tipo "o pedido terminou como cancelado?" reclamaria.
- A diferença entre 2PC e Saga não é "um funciona e o outro não": é **onde cada um cobra o preço**. O 2PC cobra em desempenho; a Saga cobra em código de compensação.

## Para praticar

1. Faça `estornar` falhar uma vez a cada três chamadas. O que acontece com o pedido cancelado?
2. Crie um laço que repete a compensação até funcionar. Para que isso seja seguro, ela precisa ser **idempotente**: executá-la duas vezes não pode estornar duas vezes. Como você garantiria isso?